# Tutorial: Análisis de Yield Curves con bonos-lib

En este notebook aprenderás a usar **bonos-lib** para calcular yield curves, hacer bootstrapping e interpolar tasas de bonos cupón cero.

## 1. Instalar la librería

In [ ]:
!pip install bonos-lib

## 2. Importar librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bonos_lib import YieldCurveCalculator

## 3. Crear datos de ejemplo

Crearemos un DataFrame con bonos cupón cero (valor nominal = 100).

In [ ]:
# Datos de bonos: días al vencimiento y precio
bonos = pd.DataFrame({
    'dias': [30, 60, 90, 180, 360],
    'precio': [99.25, 98.50, 97.75, 96.00, 92.00]
})

print("Bonos cupón cero (valor nominal = 100):")
print(bonos)
print("\nInterpretación:")
print("- A 30 días: precio 99.25 → descuento de 0.75")
print("- A 360 días: precio 92.00 → descuento de 8.00")

## 4. Calcular la Yield Curve

La yield curve muestra las tasas de rendimiento para diferentes plazos.

In [ ]:
# Crear calculator
calc = YieldCurveCalculator(nominal=100, days_year=360)

# Calcular yields
yields = calc.calculate_yields(bonos)

print("Yield Curve (Tasas de rendimiento):")
print(yields)
print("\nTasas en porcentaje:")
print(yields[['dias', 'yield']].assign(yield_pct=yields['yield'] * 100))

## 5. Visualizar la Yield Curve

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(yields['dias'], yields['yield'] * 100, 'o-', linewidth=2, markersize=8, label='Yield Curve')
plt.xlabel('Plazo (días)', fontsize=12)
plt.ylabel('Tasa de Rendimiento (%)', fontsize=12)
plt.title('Yield Curve - Bonos Cupón Cero', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Tasa mínima: {yields['yield'].min()*100:.2f}% ({yields[yields['yield']==yields['yield'].min()]['dias'].values[0]:.0f} días)")
print(f"Tasa máxima: {yields['yield'].max()*100:.2f}% ({yields[yields['yield']==yields['yield'].max()]['dias'].values[0]:.0f} días)")

## 6. Bootstrapping - Calcular Tasas Forward

El bootstrapping nos permite calcular las tasas forward (tasas futuras) a partir de las tasas spot (tasas actuales).

In [ ]:
# Realizar bootstrapping
forwards = calc.bootstrap(yields)

print("Tasas Spot y Forward:")
print(forwards)
print("\nEn porcentaje:")
display_forwards = forwards.copy()
display_forwards['spot_rate'] = display_forwards['spot_rate'] * 100
display_forwards['forward_rate'] = display_forwards['forward_rate'] * 100
print(display_forwards.round(4))

### Interpretación del Bootstrapping

- **Tasa Spot**: Tasa actual para un plazo específico (la que calculamos directamente)
- **Tasa Forward**: Tasa implícita para un período futuro

Ejemplo:
- La tasa spot a 60 días es la tasa hoy para prestar dinero por 60 días
- La tasa forward de 30-60 es la tasa implícita para prestar dinero entre el día 30 y el día 60

## 7. Interpolación - Obtener Tasas para Plazos Intermedios

In [ ]:
# Interpolar tasa para 45 días
tasa_45 = calc.interpolate(yields, 45)
print(f"Tasa para 45 días: {tasa_45*100:.4f}%")

# Interpolar para varios plazos
plazos = [15, 45, 120, 270, 300]
tasas_interpoladas = calc.interpolate_multiple(yields, plazos)

print("\nTasas interpoladas:")
for plazo, tasa in tasas_interpoladas.items():
    print(f"  {plazo:3.0f} días: {tasa*100:.4f}%")

## 8. Visualizar Yield Curve Interpolada

In [ ]:
# Crear curva interpolada detallada
from bonos_lib import LinearInterpolator

interpolator = LinearInterpolator()
curve_interpolada = interpolator.interpolate_range(yields, min_days=30, max_days=360, step=5)

plt.figure(figsize=(12, 6))

# Plotear datos originales
plt.plot(yields['dias'], yields['yield']*100, 'o', markersize=10, 
         label='Datos observados', color='red', zorder=5)

# Plotear curva interpolada
plt.plot(curve_interpolada['dias'], curve_interpolada['yield']*100, '-', 
         linewidth=2, label='Interpolación lineal', color='blue', alpha=0.7)

plt.xlabel('Plazo (días)', fontsize=12)
plt.ylabel('Tasa de Rendimiento (%)', fontsize=12)
plt.title('Yield Curve Interpolada', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 9. Análisis Completo

In [ ]:
# Análisis integral
analisis = calc.analyze(bonos)

print("=" * 50)
print("ANÁLISIS COMPLETO DE YIELD CURVE")
print("=" * 50)

print(f"\nRango de plazos: {analisis['min_days']:.0f} - {analisis['max_days']:.0f} días")
print(f"\nEstadísticas de tasas:")
print(f"  Mínima: {analisis['min_yield']*100:.4f}%")
print(f"  Máxima: {analisis['max_yield']*100:.4f}%")
print(f"  Promedio: {analisis['avg_yield']*100:.4f}%")

print(f"\nYield Curve:")
print(analisis['yields_df'].to_string())

print(f"\nTasas Forward:")
print(analisis['forwards_df'].to_string())

## 10. Ejemplo Práctico: Cargar desde CSV

In [ ]:
# Puedes cargar datos reales desde un CSV
# Simplemente asegúrate de que tenga columnas 'dias' y 'precio'

# Ejemplo: crear un CSV y cargarlo
csv_content = """dias,precio
30,99.25
60,98.50
90,97.75
180,96.00
360,92.00"""

# Guardar en archivo
with open('bonos_ejemplo.csv', 'w') as f:
    f.write(csv_content)

# Cargar desde archivo
bonos_csv = pd.read_csv('bonos_ejemplo.csv')

# Calcular yields
calc2 = YieldCurveCalculator(nominal=100)
yields_csv = calc2.calculate_yields(bonos_csv)

print("Datos cargados desde CSV:")
print(yields_csv)

## Resumen

En este tutorial aprendiste a:

1. ✅ **Instalar bonos-lib** desde PyPI
2. ✅ **Calcular Yield Curves** a partir de bonos cupón cero
3. ✅ **Hacer Bootstrapping** para obtener tasas forward
4. ✅ **Interpolar tasas** para plazos intermedios
5. ✅ **Visualizar** la curva de rendimiento
6. ✅ **Analizar** datos completos

### Próximos pasos:
- Carga tus propios datos de bonos
- Experimenta con diferentes métodos de interpolación
- Usa los datos para análisis de riesgo o valuación de instrumentos
- Integra con otros análisis financieros

Para más información, visita el repositorio: https://github.com/tu-usuario/bonos-lib